In [ ]:
# 1. Ambiente e caminhos
import os, json, datetime as dt, numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
CONJUNTO = os.environ.get("CONJUNTO_TINYML", "coleta")   # "coleta" = dados do nó | "coleta_sim" = sessões sintéticas (notebook 07b)
CONJUNTO = "coleta_sim"
SIM = CONJUNTO != "coleta"
COLETA = f"{BASE}/{CONJUNTO}"; REL = f"{COLETA}/relatorios"; os.makedirs(REL, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 10, "axes.titlesize": 10.5, "savefig.dpi": 300, "savefig.bbox": "tight"})
virgula = plt.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "\u2212"))

In [ ]:
# 14 — Manifesto de reprodutibilidade da sessão SIM006
# Cole em uma célula do Colab (ou rode como script) DEPOIS de montar o Drive.
# Não treina nem altera nada: só lê os arquivos existentes, calcula hashes,
# verifica a separação por origem e reconfere a Tabela 13 a partir das predições salvas.
# Saídas: manifesto_SIM006.json, manifesto_SIM006_arquivos.csv, manifesto_SIM006_fontes.csv

import os, json, hashlib, glob, platform, sys
import numpy as np, pandas as pd

BASE = os.environ.get("BASE_TINYML", "/content/drive/MyDrive/dissertacao_tinyml")
COLETA = f"{BASE}/coleta_sim"
ROT    = f"{COLETA}/rotulado"
MOD    = f"{BASE}/modelos/sim"
RES    = f"{BASE}/resultados_sim"
NB     = os.environ.get("NOTEBOOKS_TINYML", BASE)       # pasta onde estão os .ipynb
SAIDA  = f"{BASE}/manifesto"; os.makedirs(SAIDA, exist_ok=True)
SESSAO = "SIM006"

def sha256(p, bloco=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(bloco), b""): h.update(b)
    return h.hexdigest()

def listar(padroes):
    out = []
    for pad in padroes:
        out += [p for p in glob.glob(pad, recursive=True) if os.path.isfile(p)]
    return sorted(set(out))

problemas, M = [], {"sessao": SESSAO}

# 1. Papel das sessões e parâmetros de geração (semente) --------------------------
S = pd.read_csv(f"{COLETA}/sessoes.csv", dtype=str).fillna("")
M["papeis"] = S.groupby("papel").sessao.apply(list).to_dict()
if S.loc[S.sessao == SESSAO, "papel"].tolist() != ["teste"]:
    problemas.append(f"{SESSAO} não está marcada como teste em sessoes.csv")
par = json.load(open(f"{COLETA}/parametros_geracao.json"))
M["parametros_geracao"] = par
M["semente_geracao"] = next((par[k] for k in par if "semente" in k.lower() or "seed" in k.lower()), "NÃO ENCONTRADA")

# 2. Mapeamento dos eventos de SIM006 ---------------------------------------------
PL = pd.read_csv(f"{COLETA}/planilha_eventos.csv")
ev = PL[PL.sessao == SESSAO].copy()
ev.to_csv(f"{SAIDA}/manifesto_{SESSAO}_eventos.csv", index=False)
M["eventos"] = {"n": int(len(ev)), "por_tipo_nivel": ev.groupby(["tipo", "nivel"]).size().astype(int)
                .rename("n").reset_index().to_dict("records"),
                "com_fundo": int(ev["fundo"].astype(str).str.lower().isin(["1", "true", "sim"]).sum()) if "fundo" in ev else None}
if len(ev) != 12: problemas.append(f"planilha_eventos tem {len(ev)} eventos para {SESSAO} (esperado 12)")

# 3. Decisões rotuladas (1.998) ----------------------------------------------------
d = dict(np.load(f"{ROT}/{SESSAO}.npz"))
n_dec = int(len(d["dec_y"]))
M["decisoes"] = {"n": n_dec, "por_classe": np.bincount(d["dec_y"], minlength=4).tolist(),
                 "eventos_com_decisao": int(len(np.unique(d["dec_ev"][d["dec_ev"] > 0])))}
if n_dec != 1998: problemas.append(f"{SESSAO}.npz tem {n_dec} decisões (esperado 1.998)")

# 4. Origem dos trechos (MIMII e Bristol) e verificação de reutilização -----------
# Procura, nas pastas das sessões, um registro de origem gravado pelo 07b
# (colunas com nome de arquivo MIMII, bloco/índice Bristol e partição).
cand = listar([f"{COLETA}/**/*fonte*.csv", f"{COLETA}/**/*origem*.csv", f"{COLETA}/**/*manifest*.csv",
               f"{COLETA}/**/*clipes*.csv", f"{COLETA}/**/*trechos*.csv"])
fontes = []
for p in cand:
    t = pd.read_csv(p, dtype=str)
    if "sessao" not in t.columns:
        t["sessao"] = next((s for s in S.sessao if s in p), "")
    t["arquivo_registro"] = os.path.relpath(p, BASE); fontes.append(t)
if fontes:
    F = pd.concat(fontes, ignore_index=True)
    F.to_csv(f"{SAIDA}/manifesto_{SESSAO}_fontes.csv", index=False)
    col_id = [c for c in F.columns if any(k in c.lower() for k in ("arquivo", "clipe", "wav", "bloco", "trecho", "id"))]
    papel = dict(zip(S.sessao, S.papel))
    F["papel"] = F.sessao.map(papel)
    reuso = {}
    for c in col_id:
        x = F[[c, "papel"]].dropna().drop_duplicates()
        dup = x.groupby(c).papel.nunique()
        reuso[c] = int((dup > 1).sum())
    M["origem"] = {"registros": [os.path.relpath(p, BASE) for p in cand],
                   "colunas_identificadoras": col_id,
                   "ids_em_mais_de_uma_particao": reuso}
    if any(reuso.values()): problemas.append(f"identificadores repetidos entre partições: {reuso}")
else:
    M["origem"] = "NÃO REGISTRADA — o 07b não gravou a lista de gravações/trechos usados por sessão"
    problemas.append("origem dos trechos não registrada: acrescentar ao 07b o registro dos IDs (ver nota no fim)")

# 5. Hashes de dados, rótulos, modelos, resultados e notebooks ---------------------
arquivos = listar([f"{COLETA}/sessoes.csv", f"{COLETA}/planilha_eventos.csv", f"{COLETA}/parametros_geracao.json",
                   f"{COLETA}/{SESSAO}/**/*", f"{ROT}/{SESSAO}.npz", f"{COLETA}/linha_de_base/*",
                   f"{MOD}/*", f"{RES}/predicoes_teste.csv", f"{RES}/metricas_teste.csv", f"{RES}/ic_bootstrap.csv",
                   f"{RES}/hipoteses.json", f"{NB}/07b*.ipynb", f"{NB}/10_*.ipynb", f"{NB}/11b*.ipynb", f"{NB}/12_*.ipynb"])
A = pd.DataFrame([{"arquivo": os.path.relpath(p, BASE), "bytes": os.path.getsize(p), "sha256": sha256(p)} for p in arquivos])
A.to_csv(f"{SAIDA}/manifesto_{SESSAO}_arquivos.csv", index=False)
M["arquivos"] = int(len(A))
M["modelos"] = A[A.arquivo.str.startswith("modelos/")][["arquivo", "sha256"]].to_dict("records")

# 6. Reconferência da Tabela 13 a partir das predições salvas ---------------------
from sklearn.metrics import accuracy_score, f1_score, recall_score
R = pd.read_csv(f"{RES}/predicoes_teste.csv")
ab = {"multimodal": "Multimodal (fusão treinada)", "fusao_regra": "Multimodal (fusão por regra)",
      "lstm": "Unimodal LSTM", "cnn": "Unimodal CNN", "regra_limiar": "Regra de limiar térmico"}
T = []
for k, nome in ab.items():
    rc = recall_score(R.y, R[k], labels=[0, 1, 2, 3], average=None, zero_division=0)
    T.append([nome, accuracy_score(R.y, R[k]), f1_score(R.y, R[k], labels=[0, 1, 2, 3], average="macro", zero_division=0), *rc])
T = pd.DataFrame(T, columns=["Abordagem", "Acurácia", "F1 macro", "Recall Normal", "Recall Atenção", "Recall Crítico", "Recall Falha"]).round(3)
T.to_csv(f"{SAIDA}/tabela13_reproduzida.csv", index=False, sep=";", encoding="utf-8-sig")
M["tabela13_reproduzida"] = T.to_dict("records")
if len(R) != n_dec: problemas.append(f"predicoes_teste.csv tem {len(R)} linhas e o .npz tem {n_dec} decisões")

# 7. Ambiente ----------------------------------------------------------------------
import sklearn
try:
    import tensorflow as tf; tfv = tf.__version__
except Exception: tfv = "não importado"
M["ambiente"] = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "tensorflow": tfv, "plataforma": platform.platform()}
M["problemas"] = problemas
json.dump(M, open(f"{SAIDA}/manifesto_{SESSAO}.json", "w"), indent=1, ensure_ascii=False, default=str)

print(T.to_string(index=False))
print("\nDecisões:", n_dec, "| eventos:", M["eventos"]["n"], "| semente:", M["semente_geracao"], "| arquivos com hash:", len(A))
print("Origem:", M["origem"] if isinstance(M["origem"], str) else M["origem"]["ids_em_mais_de_uma_particao"])
print("PROBLEMAS:", problemas or "nenhum")

# NOTA — se a origem não estiver registrada, acrescente ao 07b, no ponto em que cada
# clipe MIMII e cada trecho Bristol é sorteado, uma linha como:
#   fontes.append({"sessao": s, "evento": e, "modalidade": "mimii", "arquivo": caminho_wav, "particao": part})
#   fontes.append({"sessao": s, "evento": e, "modalidade": "bristol", "bloco": id_bloco, "inicio": i0, "fim": i1, "particao": part})
# e salve pd.DataFrame(fontes).to_csv(f"{COLETA}/{s}/fontes.csv").
# Regerar com a MESMA semente deve produzir arquivos idênticos: confira pelos sha256 deste manifesto
# antes e depois; se algum hash mudar, as sessões NÃO devem ser substituídas.